# Floci Lakehouse Demo with Raspberry Pi Sense HAT, Java 25, Pi4J, Spark, Hudi, Delta Lake, and Iceberg

This notebook reads live temperature values from a Raspberry Pi **Sense HAT** using **Java 25** and **Pi4J Drivers**, then writes the readings to a local S3-compatible Floci endpoint using Apache Spark.

The demo covers:

- Reading temperature from Sense HAT
- Creating a Spark DataFrame
- Writing and reading Apache Parquet
- Writing and reading Apache Hudi
- Writing and reading Delta Lake
- Writing and reading Apache Iceberg

Expected runtime environment:

- Raspberry Pi 4, 64-bit OS
- Sense HAT attached
- I2C enabled
- Docker container running with access to `/dev/i2c-1`
- Java 25
- Jupyter Java kernel
- Floci running as `http://floci:4566`


## 1. Maven Dependencies

The Pi4J Sense HAT driver is provided by `com.pi4j:pi4j-drivers`.

This notebook also loads the lakehouse dependencies used by Spark:

- `hadoop-aws`
- `aws-java-sdk-bundle`
- `delta-spark`
- `hudi-spark3.5-bundle`
- `iceberg-spark-runtime-3.5`


In [ ]:
%maven com.pi4j:pi4j-core:4.0.1
%maven com.pi4j:pi4j-plugin-raspberrypi:4.0.1
%maven com.pi4j:pi4j-plugin-linuxfs:4.0.1
%maven com.pi4j:pi4j-plugin-gpiod:4.0.1
%maven com.pi4j:pi4j-plugin-ffm:4.0.1
%maven com.pi4j:pi4j-drivers:1.0.0

%maven org.apache.hadoop:hadoop-aws:3.3.4
%maven com.amazonaws:aws-java-sdk-bundle:1.12.262

%maven io.delta:delta-spark_2.12:3.2.0
%maven org.apache.hudi:hudi-spark3.5-bundle_2.12:1.0.2
%maven org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1


## 2. Check Java Version

This notebook is intended to run with Java 25 because `pi4j-drivers` is built for Java 25.


In [ ]:
System.out.println("Java version: " + System.getProperty("java.version"));
System.out.println("Java vendor: " + System.getProperty("java.vendor"));


## 3. Read Temperature from Sense HAT

The Sense HAT driver exposes a high-level `SenseHat` class.

`getTemperature()` returns an average of the temperature reported by the humidity sensor and pressure sensor.


In [ ]:
import com.pi4j.Pi4J;
import com.pi4j.context.Context;
import com.pi4j.drivers.hat.raspberry.SenseHat;

Context pi4j = Pi4J.newAutoContext();

SenseHat senseHat = new SenseHat(pi4j);

double currentTemperature = senseHat.getTemperature();
float currentHumidity = senseHat.getHumidity();
double currentPressure = senseHat.getPressure();

System.out.printf("Temperature: %.2f C%n", currentTemperature);
System.out.printf("Humidity: %.2f %%RH%n", currentHumidity);
System.out.printf("Pressure: %.2f hPa%n", currentPressure);


## 4. Collect Multiple Sensor Readings

For the lakehouse demo, collect a small batch of readings from the Sense HAT.


In [ ]:
import java.time.Instant;
import java.util.ArrayList;
import java.util.List;

record TemperatureReading(
    String deviceId,
    Instant timestamp,
    double temperatureCelsius,
    double humidityPercent,
    double pressureHpa
) {}

String deviceId = "raspberry-pi-4-sensehat-01";

List<TemperatureReading> readings = new ArrayList<>();

for (int i = 0; i < 10; i++) {
    readings.add(new TemperatureReading(
        deviceId,
        Instant.now(),
        senseHat.getTemperature(),
        senseHat.getHumidity(),
        senseHat.getPressure()
    ));

    Thread.sleep(1000);
}

readings.forEach(System.out::println);


## 5. Create Spark Session

The Spark session uses:

- Spark local mode
- Floci S3 endpoint: `http://floci:4566`
- S3 path-style access
- Delta Lake extensions
- Iceberg Hadoop catalog


In [ ]:
import org.apache.spark.sql.SparkSession;

SparkSession spark = SparkSession.builder()
    .appName("floci-sensehat-java-lakehouse")
    .master("local[*]")

    // S3A / Floci
    .config("spark.hadoop.fs.s3a.endpoint", "http://floci:4566")
    .config("spark.hadoop.fs.s3a.access.key", "test")
    .config("spark.hadoop.fs.s3a.secret.key", "test")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

    // Delta Lake
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension,org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

    // Iceberg
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", "s3a://iot-iceberg/warehouse")

    .getOrCreate();

spark.sparkContext().setLogLevel("WARN");

System.out.println("Spark version: " + spark.version());


## 6. Create a DataFrame from Sense HAT Readings

In [ ]:
var df = spark.createDataFrame(readings, TemperatureReading.class);

df.printSchema();
df.show(false);


## 7. Write and Read Parquet

Parquet is the physical columnar file format. Hudi, Delta Lake, and Iceberg all use Parquet as a common storage format underneath.


In [ ]:
String parquetPath = "s3a://iot-raw/sensehat/temperature_parquet";

df.write()
  .mode("overwrite")
  .parquet(parquetPath);

var parquetDf = spark.read()
  .parquet(parquetPath);

parquetDf.show(false);


## 8. Write and Read Apache Hudi

Hudi adds table-level capabilities such as commits, upserts, and incremental processing on top of Parquet files.


In [ ]:
String hudiPath = "s3a://iot-hudi/sensehat/temperature_hudi";

df.write()
  .format("hudi")
  .option("hoodie.table.name", "temperature_hudi")
  .option("hoodie.datasource.write.recordkey.field", "deviceId")
  .option("hoodie.datasource.write.precombine.field", "timestamp")
  .option("hoodie.datasource.write.partitionpath.field", "deviceId")
  .option("hoodie.datasource.write.operation", "upsert")
  .mode("overwrite")
  .save(hudiPath);

var hudiDf = spark.read()
  .format("hudi")
  .load(hudiPath);

hudiDf.show(false);


## 9. Write and Read Delta Lake

Delta Lake adds a transaction log to Parquet data and enables table operations such as updates, deletes, merges, and time travel.


In [ ]:
String deltaPath = "s3a://iot-delta/sensehat/temperature_delta";

df.write()
  .format("delta")
  .mode("overwrite")
  .save(deltaPath);

var deltaDf = spark.read()
  .format("delta")
  .load(deltaPath);

deltaDf.show(false);


## 10. Write and Read Apache Iceberg

Iceberg provides a table format with metadata, snapshots, schema evolution, partition evolution, and engine interoperability.


In [ ]:
df.createOrReplaceTempView("sensehat_readings");

spark.sql("""
CREATE TABLE IF NOT EXISTS local.default.temperature_iceberg (
  deviceId STRING,
  timestamp TIMESTAMP,
  temperatureCelsius DOUBLE,
  humidityPercent DOUBLE,
  pressureHpa DOUBLE
)
USING iceberg
""");

spark.sql("""
INSERT INTO local.default.temperature_iceberg
SELECT deviceId, timestamp, temperatureCelsius, humidityPercent, pressureHpa
FROM sensehat_readings
""");

spark.sql("SELECT * FROM local.default.temperature_iceberg").show(false);


## 11. Cleanup

Close Pi4J resources when finished.


In [ ]:
pi4j.shutdown();
spark.stop();

System.out.println("Done.");


## Troubleshooting

### Sense HAT cannot be detected

Make sure I2C is enabled on the Raspberry Pi:

```bash
sudo raspi-config
```

Then enable:

```text
Interface Options -> I2C
```

Reboot after enabling I2C.

### Docker container cannot access Sense HAT

The Jupyter container needs access to the I2C device. Add this to the `jupyter` service in `docker-compose.yml`:

```yaml
devices:
  - "/dev/i2c-1:/dev/i2c-1"
privileged: true
```

For LED matrix access you may also need framebuffer access:

```yaml
devices:
  - "/dev/i2c-1:/dev/i2c-1"
  - "/dev/fb0:/dev/fb0"
```

### Java version is not 25

Use a custom Dockerfile based on `jupyter/all-spark-notebook` and install a Java 25 JDK, then configure `JAVA_HOME`.
